In [8]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from pyspark.sql.functions import window, sum as spark_sum, count, col
import os
import shutil
import time
import pandas as pd
from datetime import datetime, timedelta

# Membuat SparkSession sesuai ketentuan
spark = SparkSession.builder \
    .appName("Tugas10-KopiNusantara") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

# Menyiapkan folder simulasi dan output
folder_stream = "tugas10_stream"
folder_output = "tugas10_output"
folder_checkpoint = "tugas10_checkpoint"

for folder in [folder_stream, folder_output, folder_checkpoint]:
    shutil.rmtree(folder, ignore_errors=True)
    os.makedirs(folder, exist_ok=True)

print("Inisialisasi SparkSession dan direktori simulasi selesai.")

Inisialisasi SparkSession dan direktori simulasi selesai.


In [9]:
# Mendefinisikan skema eksplisit untuk data pesanan Kopi Nusantara
skema_pesanan = StructType([
    StructField("pesanan_id", StringType(), True),
    StructField("nama_cabang", StringType(), True),
    StructField("nama_menu", StringType(), True),
    StructField("qty", IntegerType(), True),
    StructField("harga_satuan", IntegerType(), True),
    StructField("waktu_pesan", TimestampType(), True)
])

# Membaca data streaming (CSV) dari folder tugas10_stream
df_stream = spark.readStream \
    .format("csv") \
    .schema(skema_pesanan) \
    .option("header", True) \
    .load(folder_stream)

print("Bagian A Selesai. df_stream.isStreaming:", df_stream.isStreaming)

Bagian A Selesai. df_stream.isStreaming: True


In [14]:
# 1. Hentikan query lama jika masih ada yang aktif dengan nama yang sama
for q in spark.streams.active:
    if q.name == "agregasi_cabang":
        q.stop()

# 2. Proses transformasi & agregasi
df_stream_with_total = df_stream.withColumn("total_harga", col("qty") * col("harga_satuan"))

agregasi_cabang = df_stream_with_total.groupBy("nama_cabang").agg(
    spark_sum("total_harga").alias("total_pendapatan"),
    count("pesanan_id").alias("jumlah_pesanan")
)

# 3. Jalankan kembali streaming query
query1 = agregasi_cabang.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_cabang") \
    .start()

print("Bagian B Selesai. Query 'agregasi_cabang' aktif:", query1.isActive)

Bagian B Selesai. Query 'agregasi_cabang' aktif: True


In [15]:
# Definisi ulang readStream untuk streaming query kedua
df_stream2 = spark.readStream \
    .format("csv") \
    .schema(skema_pesanan) \
    .option("header", True) \
    .load(folder_stream) \
    .withColumn("total_harga", col("qty") * col("harga_satuan"))

# Agregasi rentang window waktu 2 menit dari kolom waktu_pesan
agregasi_waktu = df_stream2.groupBy(
    window(col("waktu_pesan"), "2 minutes")
).agg(
    spark_sum("total_harga").alias("total_pendapatan"),
    count("pesanan_id").alias("jumlah_pesanan")
)

# Menjalankan Streaming Query ke sink 'memory'
query2 = agregasi_waktu.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_waktu") \
    .start()

print("Bagian C Selesai. Query 'agregasi_waktu' aktif:", query2.isActive)

Bagian C Selesai. Query 'agregasi_waktu' aktif: True


In [18]:
waktu_sekarang = datetime.now()

# Pastikan folder bersih dari sisa run sebelumnya
shutil.rmtree(folder_stream, ignore_errors=True)
os.makedirs(folder_stream, exist_ok=True)

# === Batch 1 ===
batch_1 = pd.DataFrame({
    "pesanan_id": ["P001", "P002", "P003"],
    "nama_cabang": ["Cabang Malioboro", "Cabang Alun-Alun", "Cabang Malioboro"],
    "nama_menu": ["Kopi Tubruk", "Es Kopi Susu", "Americano"],
    "qty": [2, 1, 3],
    "harga_satuan": [15000, 20000, 18000],
    "waktu_pesan": [
        (waktu_sekarang - timedelta(minutes=3)).strftime("%Y-%m-%d %H:%M:%S"),
        (waktu_sekarang - timedelta(minutes=1)).strftime("%Y-%m-%d %H:%M:%S"),
        waktu_sekarang.strftime("%Y-%m-%d %H:%M:%S")
    ]
})
batch_1.to_csv(f"{folder_stream}/batch_1.csv", index=False)
print("=== Batch 1 Dikirim ===")
time.sleep(10)  # Dinaikkan dari 6 ke 10 detik
spark.sql("SELECT * FROM agregasi_cabang ORDER BY total_pendapatan DESC").show()

# === Batch 2 ===
batch_2 = pd.DataFrame({
    "pesanan_id": ["P004", "P005", "P006"],
    "nama_cabang": ["Cabang Borobudur", "Cabang Malioboro", "Cabang Alun-Alun"],
    "nama_menu": ["Kopi Jos", "Latte", "Es Kopi Susu"],
    "qty": [2, 2, 4],
    "harga_satuan": [12000, 25000, 20000],
    "waktu_pesan": [
        waktu_sekarang.strftime("%Y-%m-%d %H:%M:%S"),
        waktu_sekarang.strftime("%Y-%m-%d %H:%M:%S"),
        (waktu_sekarang + timedelta(minutes=1)).strftime("%Y-%m-%d %H:%M:%S")
    ]
})
batch_2.to_csv(f"{folder_stream}/batch_2.csv", index=False)
print("=== Batch 2 Dikirim ===")
time.sleep(10)
spark.sql("SELECT * FROM agregasi_cabang ORDER BY total_pendapatan DESC").show()

# === Batch 3 ===
batch_3 = pd.DataFrame({
    "pesanan_id": ["P007", "P008"],
    "nama_cabang": ["Cabang Borobudur", "Cabang Alun-Alun"],
    "nama_menu": ["V60", "Kopi Tubruk"],
    "qty": [1, 5],
    "harga_satuan": [28000, 15000],
    "waktu_pesan": [
        (waktu_sekarang + timedelta(minutes=1)).strftime("%Y-%m-%d %H:%M:%S"),
        (waktu_sekarang + timedelta(minutes=2)).strftime("%Y-%m-%d %H:%M:%S")
    ]
})
batch_3.to_csv(f"{folder_stream}/batch_3.csv", index=False)
print("=== Batch 3 Dikirim ===")
time.sleep(10)
spark.sql("SELECT * FROM agregasi_cabang ORDER BY total_pendapatan DESC").show()

# Memeriksa hasil window waktu
print("=== Hasil Agregasi Window Waktu (2 Menit) ===")
spark.sql("SELECT window.start, window.end, total_pendapatan, jumlah_pesanan FROM agregasi_waktu ORDER BY window.start").show(truncate=False)

=== Batch 1 Dikirim ===
+----------------+----------------+--------------+
|     nama_cabang|total_pendapatan|jumlah_pesanan|
+----------------+----------------+--------------+
|Cabang Malioboro|          179000|             4|
|Cabang Alun-Alun|          175000|             3|
|Cabang Borobudur|           96000|             3|
+----------------+----------------+--------------+

=== Batch 2 Dikirim ===
+----------------+----------------+--------------+
|     nama_cabang|total_pendapatan|jumlah_pesanan|
+----------------+----------------+--------------+
|Cabang Malioboro|          179000|             4|
|Cabang Alun-Alun|          175000|             3|
|Cabang Borobudur|           96000|             3|
+----------------+----------------+--------------+

=== Batch 3 Dikirim ===
+----------------+----------------+--------------+
|     nama_cabang|total_pendapatan|jumlah_pesanan|
+----------------+----------------+--------------+
|Cabang Malioboro|          179000|             4|
|Cabang 

In [19]:
# Definisi readStream ketiga untuk menulis data mentah + total_harga ke berkas Parquet
df_stream_raw = spark.readStream \
    .format("csv") \
    .schema(skema_pesanan) \
    .option("header", True) \
    .load(folder_stream) \
    .withColumn("total_harga", col("qty") * col("harga_satuan"))

# Menggunakan format Parquet dengan outputMode append
query3 = df_stream_raw.writeStream \
    .format("parquet") \
    .option("path", folder_output) \
    .option("checkpointLocation", folder_checkpoint) \
    .outputMode("append") \
    .start()

print("Bagian E Selesai. Streaming sink Parquet (query3) dimulai.")

# Mengirim Batch 4 untuk diuji pada file sink
batch_4 = pd.DataFrame({
    "pesanan_id": ["P009", "P010"],
    "nama_cabang": ["Cabang Malioboro", "Cabang Borobudur"],
    "nama_menu": ["Espresso", "Cold Brew"],
    "qty": [3, 2],
    "harga_satuan": [15000, 22000],
    "waktu_pesan": [(waktu_sekarang + timedelta(minutes=3)).strftime("%Y-%m-%d %H:%M:%S")] * 2
})
batch_4.to_csv(f"{folder_stream}/batch_4.csv", index=False)

print("Batch 4 dikirim. Menunggu Spark menulis ke Parquet...")
time.sleep(8)

# Verifikasi file Parquet yang dibuat
print("Isi direktori 'tugas10_output':")
print([f for f in os.listdir(folder_output) if f.endswith(".parquet") or f.startswith("part-")])

# Membaca kembali berkas Parquet sebagai DataFrame Batch biasa
print("\nHasil pembacaan DataFrame batch dari Parquet:")
df_parquet_result = spark.read.parquet(folder_output)
df_parquet_result.show(truncate=False)

Bagian E Selesai. Streaming sink Parquet (query3) dimulai.
Batch 4 dikirim. Menunggu Spark menulis ke Parquet...
Isi direktori 'tugas10_output':
['part-00000-e359e684-b60a-4af8-9dcb-6aafceada91d-c000.snappy.parquet']

Hasil pembacaan DataFrame batch dari Parquet:


[Stage 19:>                                                         (0 + 1) / 1]

+----------+----------------+------------+---+------------+-------------------+-----------+
|pesanan_id|nama_cabang     |nama_menu   |qty|harga_satuan|waktu_pesan        |total_harga|
+----------+----------------+------------+---+------------+-------------------+-----------+
|P001      |Cabang Malioboro|Kopi Tubruk |2  |15000       |2026-11-11 23:21:04|30000      |
|P002      |Cabang Alun-Alun|Es Kopi Susu|1  |20000       |2026-11-11 23:23:04|20000      |
|P003      |Cabang Malioboro|Americano   |3  |18000       |2026-11-11 23:24:04|54000      |
|P004      |Cabang Borobudur|Kopi Jos    |2  |12000       |2026-11-11 23:24:04|24000      |
|P005      |Cabang Malioboro|Latte       |2  |25000       |2026-11-11 23:24:04|50000      |
|P006      |Cabang Alun-Alun|Es Kopi Susu|4  |20000       |2026-11-11 23:25:04|80000      |
|P009      |Cabang Malioboro|Espresso    |3  |15000       |2026-11-11 23:27:04|45000      |
|P010      |Cabang Borobudur|Cold Brew   |2  |22000       |2026-11-11 23:27:04|4

In [20]:
# Hentikan semua query dan SparkSession di akhir notebook
query1.stop()
query2.stop()
query3.stop()
spark.stop()

print("Seluruh streaming query dihentikan dan SparkSession berhasil ditutup.")

Seluruh streaming query dihentikan dan SparkSession berhasil ditutup.
